# Late Interaction Retrieval - 01: Why late interaction exists

> **MLCourse - Agentic AI - Advanced RAG - Module 15**

Every retriever you have built so far in this track has made one of two bets.

**Bet 1 - the bi-encoder (ordinary vector search).** Squash a whole passage
into a *single* vector, ahead of time. Squash the query into a single vector.
Compare them with one dot product. This is what `01_hybrid_search` and every
Chroma collection in this track does. It is gloriously fast, because the
document side is precomputed once and never touched again.

**Bet 2 - the cross-encoder (reranking).** Glue the query and one document
together into a single input, run a transformer over the pair, and let every
query token attend to every document token. This is
[`11_reranking`](../11_reranking/README.md). It is far more accurate, and it
is *impossible* to precompute, because the answer depends on both halves at
once. You can afford it on 50 candidates, never on a million.

**Late interaction is the third bet, and it is the interesting one.** Keep
one vector *per token* instead of one per passage. Precompute the document
tokens offline, exactly like a bi-encoder. Then, at query time, do the
interaction cheaply - not with a transformer, but with a small matrix of dot
products and a `max`. That operator is called **MaxSim**, and the family of
models built around it is named after its first member, **ColBERT**
(*Contextualized Late Interaction over BERT*, Khattab & Zaharia, 2020).

### What you will learn in this module

1. What a single vector throws away, and why that costs you recall.
2. How to get per-token embeddings out of a local model.
3. The MaxSim scoring function, implemented from scratch.
4. Where late interaction sits on the quality/cost curve, measured.
5. What it costs in **storage** - the reason it is not the default.

### A note on what we are building

The real ColBERT implementations (`pylate`, `colbert-ai`, `RAGatouille`) ship
a model that was *trained* for late interaction, plus a specialised index
(PLAID) that makes MaxSim fast over millions of documents.

**Installing `pylate` into this course's environment downgrades `torch`,
`transformers` and `sentence-transformers`, breaking other modules.** So this
module is an honest **teaching reimplementation**: we take token embeddings
from an ordinary `sentence-transformers` bi-encoder and apply the MaxSim
operator to them by hand, in NumPy.

Be clear about what that does and does not give you:

- ✅ The **scoring mechanism** is the real one. MaxSim is MaxSim.
- ✅ The **storage and latency arithmetic** is the real one.
- ❌ The **quality numbers are not** what a real ColBERT gets. A trained
  late-interaction model learns token embeddings *for* this operator; a
  bi-encoder's token embeddings are a by-product of training for a pooled
  vector. We are borrowing them.

Every time this module reports a measurement, it says which of those two
categories it is in.

### Setup: imports and the shared toy corpus


In [ ]:
# Nothing in this module calls an LLM. Everything here is a local embedding
# model plus NumPy, so every number you see is reproducible and free.

import time
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2 is the same small local model used across this track.
# It is a *bi-encoder*, and in the next notebook we will make it behave like
# a late-interaction model by asking it for its per-token output instead of
# its pooled sentence vector.
MODEL = SentenceTransformer("all-MiniLM-L6-v2")
print("model loaded:", MODEL.get_embedding_dimension(), "dims per token")

# A deliberately small corpus. Passage D0 is the true answer to our question;
# the others are near-misses of the kind that fool single-vector retrieval.
CORPUS = [
    "The Eiffel Tower was completed in 1889 and stands on the Champ de Mars in Paris.",
    "Paris is the capital and most populous city of France, on the river Seine.",
    "France is a country in Western Europe with several overseas territories.",
    "The capital of Italy is Rome, a city famous for the Colosseum and the Forum.",
    "Berlin became the capital of a reunified Germany in 1990.",
    "Many capital cities grew up around a river crossing or a defensible hill.",
    "French cuisine is celebrated worldwide; Lyon is often called its capital.",
    "The Louvre, in central Paris, is the world's most-visited art museum.",
]

QUERY = "what is the capital city of France"

print(f"\ncorpus: {len(CORPUS)} passages")
print(f"query : {QUERY!r}")
print(f"\nThe passage that actually answers it is D1:\n  {CORPUS[1]}")


### 1. What a single vector throws away

A pooled sentence embedding is an **average** (mean pooling) of the token
embeddings. Averaging is lossy in a specific way: it produces one point that
is "about" the whole passage, so a passage about *many* things ends up
somewhere in the middle, near nothing in particular.

That matters because a query is usually about **one** thing. You want to
know whether *some part* of the passage matches, not whether the passage's
centre of gravity does.

Let's see the pooled vectors first - the baseline every other notebook in
this track uses.

### Baseline: ordinary single-vector (bi-encoder) retrieval


In [ ]:
# normalize_embeddings=True makes the dot product equal to cosine similarity,
# which keeps the arithmetic below simple.
doc_vecs = MODEL.encode(CORPUS, normalize_embeddings=True)
query_vec = MODEL.encode(QUERY, normalize_embeddings=True)

print("one vector per passage:", doc_vecs.shape, " <- 8 passages, 384 dims each")

# Cosine similarity = a single dot product per passage.
dense_scores = doc_vecs @ query_vec

print("\nSingle-vector ranking:")
for rank, i in enumerate(np.argsort(-dense_scores), start=1):
    marker = "  <== the true answer" if i == 1 else ""
    print(f"  {rank}. [{dense_scores[i]:.4f}] D{i}: {CORPUS[i][:64]}...{marker}")


### Reading that ranking honestly

The bi-encoder got this right. D1 is ranked first, and comfortably. **Say so
plainly** - a module that only ever shows its baseline failing is selling you
something. On eight passages with a clearly-worded query, ordinary vector
search is entirely adequate, and if that is your situation you should stop
reading and keep your cheap index.

What is worth looking at is **ranks 2 and 3**, because that is where the
weakness shows:

- **D2** ("France is a country in Western Europe...") scores 0.64. It does
  not answer the question at all. It is merely *about France*.
- **D6** ("French cuisine... Lyon is often called its capital") scores 0.58,
  beating three passages that are more genuinely relevant. It contains
  "capital" in an unrelated, figurative sense, and the pooled vector has no
  way to know that.

Both are topically adjacent and semantically wrong, and both crowd the top of
the list. That is the characteristic bi-encoder failure: a single vector
encodes **what a passage is about**, not **which of its words lined up with
which of yours**.

On 8 documents a near-miss at rank 2 is harmless. On 8 million documents, the
same effect is why the passage you actually needed sits at rank 400 and never
reaches your reranker - the failure is one of **recall at the first stage**,
which is precisely the stage late interaction targets.

The information that would have separated D1 from D6 - *where* the match
happened - was averaged away before the search ever ran. That is exactly the
information a MaxSim score keeps.

### 2. The three families, side by side

The distinction that actually matters is **when the query and the document
are allowed to interact**.

| | Bi-encoder | **Late interaction** | Cross-encoder |
|---|---|---|---|
| Covered in | `01_hybrid_search` | **this module** | [`11_reranking`](../11_reranking/README.md) |
| Doc representation | 1 vector | **1 vector per token** | none - reprocessed each time |
| Interaction happens | never (just a dot product at the end) | **at query time, over precomputed vectors** | inside the transformer, over the raw pair |
| Precomputable offline? | ✅ fully | **✅ fully** | ❌ never |
| Cost to score N docs | N dot products | **N small matrix multiplies** | N transformer forward passes |
| Typical use | first-stage retrieval, millions of docs | **first-stage retrieval with better precision** | reranking a shortlist of ~50 |
| Storage per passage | ~1.5 KB | **~50-200 KB** | 0 (nothing stored) |

The row that explains everything is **"precomputable offline?"**.

- The cross-encoder is accurate because it lets the query and the document
  interact *early* - before any pooling. That is also precisely why it cannot
  be precomputed: there is nothing to store until the query arrives.
- The bi-encoder is fast because it interacts *late* and *cheaply*, but by
  then both sides have been crushed to a single point.
- Late interaction interacts **late, like a bi-encoder, but richly**. The
  document side is still fully precomputable. The interaction is still just
  arithmetic. But it happens between *sets of token vectors*, so it can still
  see which word matched which.

The name is literal: the interaction is *late* (after encoding, at query
time), as opposed to a cross-encoder's *early* interaction.

> **This is not a replacement for module 11.** A cross-encoder still wins on
> raw accuracy. Late interaction competes with the *first* stage - it is a
> better recall engine, not a better reranker. The two compose: retrieve with
> late interaction, then rerank the shortlist with a cross-encoder.

### 3. A first look at the thing we are going to build

Before implementing MaxSim properly in the next notebook, here is the shape
of it, so the rest of the module has something concrete to refer back to.

Ask the model for **token embeddings** rather than a pooled sentence vector.
Instead of one 384-dim point per passage, you get a matrix.

### Token embeddings: the raw material of late interaction


In [ ]:
# output_value="token_embeddings" tells sentence-transformers to hand back the
# per-token output of the transformer, BEFORE the pooling layer averages it
# into a single sentence vector. That pooling step is the lossy one.
tok_query = MODEL.encode(QUERY, output_value="token_embeddings")
tok_doc1 = MODEL.encode(CORPUS[1], output_value="token_embeddings")

print(f"pooled query vector : shape {query_vec.shape}   <- 1 point")
print(f"query token matrix  : shape {tuple(tok_query.shape)}  <- one row per token")
print(f"doc D1 token matrix : shape {tuple(tok_doc1.shape)}  <- one row per token")

# Let's see the actual tokens, so the rows are not anonymous.
q_tokens = MODEL.tokenizer.convert_ids_to_tokens(MODEL.tokenizer(QUERY)["input_ids"])
d_tokens = MODEL.tokenizer.convert_ids_to_tokens(MODEL.tokenizer(CORPUS[1])["input_ids"])
print(f"\nquery tokens ({len(q_tokens)}): {q_tokens}")
print(f"\ndoc tokens   ({len(d_tokens)}): {d_tokens}")


Note `[CLS]` and `[SEP]`. These are the transformer's structural markers, not
content. Real ColBERT keeps `[CLS]` (it repurposes it) but the important
practical point for us is that **every token becomes a row**, and the number
of rows is why storage explodes - a point notebook 04 measures.

### A preview of MaxSim (the full version is notebook 02)


In [ ]:
def preview_maxsim(q_tokens_mat, d_tokens_mat):
    """For each QUERY token, find its best-matching DOC token; sum those bests."""
    q = torch.nn.functional.normalize(q_tokens_mat, dim=-1).numpy()
    d = torch.nn.functional.normalize(d_tokens_mat, dim=-1).numpy()
    sim = q @ d.T             # every query token vs every doc token
    return sim.max(axis=1).sum(), sim   # the "max" then the "sum"

score, sim_matrix = preview_maxsim(tok_query, tok_doc1)

print(f"MaxSim(query, D1) = {score:.4f}")
print(f"the similarity matrix behind it: {sim_matrix.shape}"
      f"  ({len(q_tokens)} query tokens x {len(d_tokens)} doc tokens)")

print("\nWhich doc token did each query token match best?")
for qi, qt in enumerate(q_tokens):
    best = int(sim_matrix[qi].argmax())
    print(f"  {qt:<10} -> {d_tokens[best]:<10} (sim {sim_matrix[qi, best]:.3f})")


### This is the payoff, and it is worth staring at

That last table is something a single-vector search **structurally cannot
produce**. The score is no longer an opaque number; it decomposes into "this
query word was answered by that document word."

Notice `france -> france` and `capital -> capital` lining up. Notice which
tokens found a genuine match and which merely settled for the least-bad
option available. When we score the distractor passages the same way in
notebook 03, the difference will be visible token by token: a passage that
merely mentions "capital" in another sense contributes a *weaker* max for
that token, and the sum reflects it.

That is also why late interaction is unusually **debuggable** - a rare
property in retrieval. When a bi-encoder returns a bad result you get a
number and a shrug. When MaxSim returns a bad result you can point at the
row responsible.

### 4. Why is this not simply the default, then?

Because of one number: **you are storing a matrix where you used to store a
vector.**

A rough back-of-the-envelope, made precise in notebook 04:

- Bi-encoder: 384 floats per passage.
- Late interaction: 384 floats × ~100 tokens per passage ≈ **100× more**.

Real ColBERT fights this hard - dimension reduction to 128 or fewer, then
aggressive quantization down to ~2 bits per dimension. Even so it remains
substantially larger than a single-vector index, and the query-time work
grows too, because you are now doing a small matrix multiply per candidate
instead of one dot product.

So the honest positioning is: late interaction buys **precision at
first-stage recall time**, and pays for it in **disk and index complexity**.
Whether that trade is worth it depends on whether your failure mode is
"the right document was never retrieved" (it helps a lot) or "the right
document was retrieved but ranked 8th" (a cross-encoder reranker is cheaper).

### Key takeaways

- A pooled sentence vector averages away **which token matched which** - the
  single most useful signal for judging relevance.
- **Late interaction** keeps one vector per token, precomputes the document
  side offline like a bi-encoder, and does a cheap interaction at query time.
- **MaxSim** is that interaction: for every query token, take its best match
  among the document tokens, then sum. Max, then sum.
- It sits **between** the bi-encoder and the cross-encoder on both quality
  and cost - but unlike a cross-encoder it is a **first-stage retriever**,
  not a reranker. See [`11_reranking`](../11_reranking/README.md) for that.
- Its scores are **decomposable and debuggable**, token by token.
- Its cost is **storage**, and that cost is large.

**Next:** `02_token_embeddings_and_maxsim.ipynb` - building MaxSim properly,
including the padding and normalisation details this preview glossed over.